# Imports

In [ ]:
!pip install torch_geometric

In [ ]:
import torch
import os
from torch_geometric.data import Data


In [ ]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


# Experiment Manager

In [ ]:
class ExperimentManager:
    def __init__(self, log_file="./logs/experiments_log.csv", model_dir="./saved_models"):
        self.log_file = log_file
        self.model_dir = model_dir
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

    def log_experiment(self, model_config=None, model_name=None, params=None, metrics=None, model_object=None):
        """
        Record an experiment in CSV format and optionally save the model.

        model_config: Recommended dict:
        - model_name (str)
        - type (str)
        - model_params (dict) # ONLY model hyperparameters
        - prob_threshold (float) # Decision threshold for probabilities (for metrics computation)
        - data_params (dict) # Optional
        - extra_params (dict) # Optional

        model_name: Name of the model (str)

        params: (legacy) Extra dict (for compatibility)
        metrics: Dict with results
        model_object: Model object (for saving)
        """

        if metrics is None:
            metrics = {}
        if params is None:
            params = {}

        # Timezone Argentina
        tz = timezone(timedelta(hours=-3))
        now = datetime.now(tz)

        # Input
        if model_config is not None:
            mname = model_config.get("model_name", model_name)
            mtype = model_config.get("type", None)
            model_params = model_config.get("model_params", {})
            threshold = model_config.get("prob_threshold", None)
            data_params = model_config.get("data_params", {})
            extra_params = model_config.get("extra_params", {})
        else:
            # legacy mode
            mname = model_name
            mtype = params.get("type", None)
            threshold = params.get("prob_threshold", None)
            model_params = params
            data_params = {}
            extra_params = {}

        # Create record
        entry = {
            "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
            "model_name": mname,
        }
        if mtype is not None:
            entry["type"] = mtype
        if threshold is not None:
            entry["prob_threshold"] = threshold

        # Prefijos para evitar colisiones de nombres
        entry.update({f"hp_{k}": v for k, v in (model_params or {}).items()})
        entry.update({f"data_{k}": v for k, v in (data_params or {}).items()})
        entry.update({f"extra_{k}": v for k, v in {**extra_params, **params}.items()
                      if k not in ("type", "prob_threshold")})

        entry.update(metrics)

        # Save in csv
        df_new = pd.DataFrame([entry])
        if os.path.exists(self.log_file):
            df_new.to_csv(self.log_file, mode="a", header=False, index=False)
        else:
            df_new.to_csv(self.log_file, mode="w", header=True, index=False)

        print(f"Experiment recorded in {self.log_file}")

        # Save model
        if model_object is not None:
            metric_key = "AUC-PR" if "AUC-PR" in metrics else ("F1" if "F1" in metrics else None)
            metric_val = metrics.get(metric_key, 0) if metric_key else 0
            safe_key = metric_key or "metric"
            filename = f"{mname}_{now.strftime('%Y%m%d_%H%M')}_{safe_key}_{float(metric_val):.4f}"

            if "sklearn" in str(type(model_object)):
                import joblib
                joblib.dump(model_object, os.path.join(self.model_dir, f"{filename}.joblib"))
            else:
                import torch
                torch.save(model_object.state_dict(), os.path.join(self.model_dir, f"{filename}.pth"))

            print(f"Saved model: {filename}")


# Global instance
exp_manager = ExperimentManager()

# Load data

In [ ]:
import torch
import os
from torch_geometric.data import Dataset

class NF_IDS_Dataset(Dataset):
    def __init__(self, root_dir, split='train'):
        # root_dir: (eg: "./dataset_processed")
        # split: 'train', 'val' or 'test'
        self.split_dir = os.path.join(root_dir, split)
        super().__init__(self.split_dir)

        # List files ordered numerically to respect the time
        self.files = sorted(
            [f for f in os.listdir(self.split_dir) if f.endswith('.pt')],
            key=lambda x: int(x.split('_')[1].split('.')[0])
        )

    def len(self):
        return len(self.files)

    def get(self, idx):
        data = torch.load(
            os.path.join(self.split_dir, self.files[idx]),
            weights_only=False # to allow loading complex graph objects
        )
        return data

# Models

## Static GNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv


class StaticGNN(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, out_dim, dropout):
        super(StaticGNN, self).__init__()

        # LAYER 1: Input -> Hidden
        # Takes info of immediate neighbors
        self.gnn1 = GATv2Conv(
            in_channels=node_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,        # Use 2 attention heads for added robustness
            concat=False    # Averaged the printheads to maintain a 64 dimension
        )

        # LAYER 2: Hidden -> Hidden
        # Takes info about neighbors of neighbors (Context of 2 jumps)
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False
        )

        # EDGE CLASSIFIER
        # Input: U-Node (64) + V-Node (64) + Edge Features (32)
        self.classifier = nn.Sequential(
            nn.Linear(2 * hidden_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout), # Dropout to avoid overfitting
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, out_dim)
        )


    def forward(self, x, edge_index, edge_attr):
        # Check empty graph
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # 1. Fist gnn layer
        h1 = self.gnn1(x, edge_index, edge_attr=edge_attr)
        h1 = F.elu(h1)

        # 2. Second gnn layer
        # Note: passed h1 as node features, but used the SAME graph (edge_index)
        h2 = self.gnn2(h1, edge_index, edge_attr=edge_attr)
        h2 = F.elu(h2)

        # 3. Edge Classification
        # Take the final embeddings (h2) of the source and destination nodes
        src, dst = edge_index

        # Concatenate: [Context_SrcIP, Context_DstIP, Edge_Attributes]
        edge_rep = torch.cat([h2[src], h2[dst], edge_attr], dim=1)

        # Output
        return self.classifier(edge_rep)

## ST-GNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 64
OUT_DIM = 1     # Binary output (logit): >0 is attack, <0 is benign


class ST_GNN(nn.Module):
    def __init__(self):
        super(ST_GNN, self).__init__()

        # LAYER 1 (Spatial): Input -> Hidden
        self.gnn1 = GATv2Conv(
            in_channels=NODE_DIM,
            out_channels=HIDDEN_DIM,
            edge_dim=EDGE_DIM,
            heads=2,
            concat=False # Averaged the printheads to maintain a 64 dimension
        )

        # LAYER 2 (Spatial): Hidden -> Hidden
        self.gnn2 = GATv2Conv(
            in_channels=HIDDEN_DIM,
            out_channels=HIDDEN_DIM,
            edge_dim=EDGE_DIM,
            heads=1,
            concat=False
        )

        # TEMPORAL LAYER: GRU
        self.gru = nn.GRUCell(input_size=HIDDEN_DIM, hidden_size=HIDDEN_DIM)

        # EDGE CLASSIFIER
        # Input: story_U (64) + story_V (64) + edge_dim (32)
        self.classifier = nn.Sequential(
            nn.Linear(2 * HIDDEN_DIM + EDGE_DIM, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, OUT_DIM)
        )

        # Memory: Dictionary {global_id: hidden_state_tensor}
        self.node_memory = {}

    def get_memory(self, ids, device):
        """Restore previous states or return zeros if it is a new node"""
        mem_list = []
        for i in ids:
            i = i.item()
            if i in self.node_memory:
                mem_list.append(self.node_memory[i])
            else:
                mem_list.append(torch.zeros(HIDDEN_DIM, device=device))
        return torch.stack(mem_list)

    def update_memory(self, ids, h_new):
        """Save the new states in the global dictionary"""
        h_detached = h_new.detach() # Detach for Truncated BPTT
        for idx, global_id in enumerate(ids):
            self.node_memory[global_id.item()] = h_detached[idx]

    def detach_all_memory(self):
        """Call every 10 steps ("sequential batch") to clean old computational graphs"""
        for k in self.node_memory:
            self.node_memory[k] = self.node_memory[k].detach()

    def forward(self, x, edge_index, edge_attr, global_ids):
        # 0. Check empty graphs
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # 1. Recover Memory (H_{t-1})
        h_prev = self.get_memory(global_ids, x.device)

        # 2. Spatial (GNNs) -> Z_t
        # Layer 1
        z = self.gnn1(x, edge_index, edge_attr=edge_attr)
        z = F.elu(z)
        # Layer 2
        z = self.gnn2(z, edge_index, edge_attr=edge_attr)
        z = F.elu(z)

        # 3. Temporal (GRU) -> H_t
        h_current = self.gru(z, h_prev)

        # 4. Update Memory
        self.update_memory(global_ids, h_current)

        # 5. Edge Classification (using the temporarily enriched hidden state)
        src, dst = edge_index
        edge_rep = torch.cat([h_current[src], h_current[dst], edge_attr], dim=1)

        return self.classifier(edge_rep)

# Functions

This handles the complex logic: Truncated Backpropagation for ST-GNN and detailed metric calculations (including False Positives).

## Metrics

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, fbeta_score, average_precision_score, roc_auc_score
import numpy as np

def calculate_metrics(y_true, y_pred_logits, prob_threshold=0.5):
    # Convert logits to binary (Sigmoid > 0.5 is equivalent to Logit > 0)
    probs = torch.sigmoid(torch.tensor(y_pred_logits))
    preds = (probs > prob_threshold).long().numpy()

    # Calculate standard metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)

    # Calculate other metrics
    f2 = fbeta_score(y_val, preds, beta=2)
    ap = average_precision_score(y_val, y_probs)
    roc = roc_auc_score(y_val, y_probs)

    # Calculate False Positive Rate (FPR)
    # TN = True Negative, FP = False Positive
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2, "AUC-PR": ap, "AUC-ROC": roc,
        "FPR": fpr, "TP": tp, "FP": fp, "TN": tn, "FN": fn, "Total_Flows": len(y_true)
    }



## Train and eval

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, is_temporal=False, batch_steps=10):
    model.train()
    total_loss = 0
    steps = 0  # Count of valid graphs processed (not empty)

    # Loss accumulator for Truncated Backprop
    batch_loss = 0

    # Iterate over the Loader
    # batch_idx helps to know if we are on the last element
    for batch_idx, data in enumerate(loader):
        data = data.to(device)

        # If it is an empty graph, skip
        if data.x.shape[0] == 0:
            continue

        # Forward
        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        # Loss calculation
        loss = criterion(out.view(-1), data.y)
        batch_loss += loss
        steps += 1

        # Backpropagation:
        # 1. Each 'batch_steps' valid steps (e.g., every 10 graphs)
        # 2. Or if it is in the LAST batch of the loader (so as not to lose the remainder)
        is_last_batch = (batch_idx == len(loader) - 1)

        if (steps > 0 and steps % batch_steps == 0) or is_last_batch:
            # Only update if there's something accumulated
            if batch_loss > 0:
                optimizer.zero_grad()
                batch_loss.backward()
                optimizer.step()

                # Reset
                total_loss += batch_loss.item()
                batch_loss = 0

                # IMPORTANT for ST-GNN: Detach memory
                if is_temporal:
                    model.detach_all_memory()

    # Average loss per valid step
    return total_loss / steps if steps > 0 else 0

@torch.no_grad()
def evaluate(model, loader, criterion, device, prob_threshold, is_temporal=False):
    model.eval()
    all_preds = []
    all_trues = []
    total_loss = 0
    steps = 0

    # Clear the memory before starting the evaluation (only if it's temporal)
    if is_temporal:
        model.node_memory = {}

    # Don't need enumerate here because we're not doing backprop
    for data in loader:
        data = data.to(device)

        if data.x.shape[0] == 0: continue

        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        loss = criterion(out.view(-1), data.y)
        total_loss += loss.item()

        all_preds.extend(out.view(-1).cpu().numpy())
        all_trues.extend(data.y.cpu().numpy())
        steps += 1

    metrics = calculate_metrics(all_trues, all_preds, prob_threshold)
    metrics["Loss"] = total_loss / steps if steps > 0 else 0
    return metrics

## Save checkpoint

In [ ]:
def save_checkpoint(model, optimizer, epoch, metrics, filename):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics
    }
    torch.save(checkpoint, filename)
    print(f"Saved model: {filename}")


MODEL_DIR = "./saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

## pos_weight

In [ ]:
def check_class_balance(loader, name="Loader"):
    total_benign = 0
    total_attack = 0

    print(f"--- Analyzing {name} ---")
    for data in loader:
        # data.y es [1] o [N]
        y = data.y.view(-1)
        n_ones = y.sum().item()
        n_zeros = y.shape[0] - n_ones

        total_attack += n_ones
        total_benign += n_zeros

    print(f"Benign: {int(total_benign)}")
    print(f"Attack : {int(total_attack)}")
    if total_attack > 0:
        ratio = total_benign / total_attack
        print(f"Ratio: 1 attack for every {ratio:.1f} benign")
        return torch.tensor([ratio])
    else:
        print("ALERT: There are no attacks on this dataset.")
        return None



# Auxiliary

In [ ]:
ROOT_PATH = "./dataset_processed"

In [ ]:
from torch_geometric.loader import DataLoader

# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)

In [ ]:
pos_weight_value = check_class_balance(train_loader, "TRAIN")
print(f"pos_weight_value:{pos_weight_value}\n")
check_class_balance(val_loader, "VAL")

# Main

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 5
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.001

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 64
OUT_DIM = 1     # Binary output (logit): >0 is attack, <0 is benign
DROPOUT = 0.2

PROB_THRESHOLD = 0.5


print(f"Using device: {DEVICE}")


In [ ]:
model_config = {
    "model_name": "StaticGNN",
    "type": "GNN_traditional",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "out_dim": OUT_DIM,
        "dropout": DROPOUT
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": pos_weight_value,
        "learning_rate": LR
    }
}


## Train static

In [ ]:
import copy

static_model = StaticGNN(**model_config['model_params']).to(DEVICE)

weight = pos_weight_value.to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=weight)

opt_static = torch.optim.Adam(static_model.parameters(), lr=LR)


best_aucpr_static = 0.0
best_wts_static = copy.deepcopy(model.state_dict())
best_metrics_static = {}

for epoch in range(EPOCHS):
    # Note: batch_steps here acts as a simple "gradient accumulation"
    loss = train_epoch(static_model, train_loader, opt_static, criterion, DEVICE, is_temporal=False, batch_steps=BATCH_STEPS)
    metrics = evaluate(static_model, val_loader, criterion, DEVICE, is_temporal=False)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f}")

    if metrics['AUC-PR'] > best_aucpr_static:
        best_aucpr_static = metrics['AUC-PR']
        best_metrics_static = metrics
        best_wts_static = copy.deepcopy(static_model.state_dict())
        print(f"Best AUC-PR: {best_aucpr_static:.4f}")


static_model.load_state_dict(best_wts_static)
exp_manager.log_experiment(model_config=model_config, metrics=best_metrics_static, model_object=static_model)

print(f"\nBest AUC-PR for Static GNN obtained: {best_aucpr_static:.4f}")

## Train ST-GNN

In [ ]:
st_model = ST_GNN().to(DEVICE)

weight = pos_weight_value.to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=weight)

opt_st = torch.optim.Adam(st_model.parameters(), lr=0.001)

best_f1_st = 0.0
for epoch in range(EPOCHS):
    loss = train_epoch(st_model, train_loader, opt_st, criterion, DEVICE, is_temporal=True, batch_steps=BATCH_STEPS)
    metrics = evaluate(st_model, val_loader, criterion, DEVICE, is_temporal=True)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val F1: {metrics['F1']:.4f} | Val FPR: {metrics['FPR']:.4f}")

    if metrics['F1'] > best_f1_st:
        best_f1_st = metrics['F1']
        save_path = os.path.join(MODEL_DIR, 'best_st_gnn.pth')
        save_checkpoint(st_model, opt_st, epoch, metrics, save_path)
        print(f"Best F1: {best_f1_st:.4f}")

print(f"\nBest F1 ST-GNN obtained: {best_f1_st:.4f}")

